# Capstone - Small Business Finance Assistant (Start)
**Day 3 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/capstone/start/capstone.ipynb)

**Goal:** Build a retrieval-augmented assistant that answers small-business
finance questions over a mock QuickBooks and TurboTax help knowledge base,
grounded in the retrieved help articles. Everyone finishes the base assistant;
then pick up the tiered stretch goals as time allows.

All data in `capstone/data/` is synthetic mock data. Look for the `# TODO`
markers and fill them in. The `solution/` notebook has the full answer.
Retrieval runs offline; generation calls an LLM API through the shared client.
Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this capstone (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers faiss-cpu litellm smolagents peft accelerate transformers torch python-dotenv

## Setup: shared client and data path
This finds the `shared/` package and the `capstone/data/` folder locally, or
clones the companion repo when running on Colab. It also makes sure an API key
is set. You do not need to change this cell.

In [ ]:
import os
import subprocess
import sys

import torch  # noqa: F401  (import torch before faiss to avoid an OpenMP clash on macOS)


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for shared/ and data.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)
DATA_DIR = os.path.join(root, "capstone", "data")

from shared import get_completion, get_model, show_config

if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## 1. Load the knowledge base
Read every Markdown help article from `capstone/data/`. Each file is one
source document the assistant is allowed to use.

In [ ]:
import glob

documents = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.md"))):
    with open(path, encoding="utf-8") as f:
        documents.append({"source": os.path.basename(path), "text": f.read()})
print(f"Loaded {len(documents)} help articles:",
      ", ".join(d["source"] for d in documents))

## 2. Chunk, embed, and index
Split each article into overlapping chunks, embed them with a small local
model, and index the vectors with FAISS. Normalized vectors make inner-product
search equal to cosine similarity.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


def chunk_text(text, chunk_size=500, overlap=60):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def build_index(docs, model, chunk_size=500, overlap=60):
    chunks = []
    for doc in docs:
        for piece in chunk_text(doc["text"], chunk_size, overlap):
            chunks.append({"source": doc["source"], "text": piece})
    # TODO: Embed every chunk text with model.encode(..., normalize_embeddings=True),
    # convert to a float32 numpy array, build a faiss.IndexFlatIP of the right
    # dimension, and add the vectors.
    # Hint: vecs = np.array(model.encode([c["text"] for c in chunks],
    #                       normalize_embeddings=True), dtype="float32")
    #       idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    vecs = ...
    idx = ...
    return chunks, idx


embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunks, index = build_index(documents, embedder)
print(f"Indexed {index.ntotal} chunks.")

## 3. Retrieve and answer, grounded
Retrieve the top chunks for a question, then answer using only that context.
If the context does not contain the answer, the assistant says so and cites
the source article otherwise.

In [ ]:
SYSTEM = (
    "You are a small-business finance assistant. Answer using ONLY the provided "
    "context from QuickBooks and TurboTax help articles. If the context does not "
    "contain the answer, say you do not have that information. Cite the source "
    "file in brackets, for example [quickbooks_invoicing.md]."
)


def retrieve(query, model, idx, chunk_list, k=3):
    q = np.array(model.encode([query], normalize_embeddings=True), dtype="float32")
    # TODO: Search the index for the top k chunks and return them with scores.
    # Hint: scores, ids = idx.search(q, k)
    scores, ids = ...
    hits = []
    for score, i in zip(scores[0], ids[0]):
        hit = dict(chunk_list[i])
        hit["score"] = float(score)
        hits.append(hit)
    return hits


def rag_answer(query, k=3):
    hits = retrieve(query, embedder, index, chunks, k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    # TODO: Build the messages with the SYSTEM prompt and a user message holding
    # the context and question, then call get_completion at tier="mini",
    # temperature=0.0. Return the answer and the hits.
    # Hint: messages = [{"role": "system", "content": SYSTEM},
    #                   {"role": "user",
    #                    "content": f"Context:\n{context}\n\nQuestion: {query}"}]
    #       return get_completion(messages, tier="mini", temperature=0.0), hits
    messages = ...
    return get_completion(messages, tier="mini", temperature=0.0), hits


DEMO_QUESTIONS = [
    "What is the default payment term on a QuickBooks invoice?",
    "How much does QuickBooks Payroll Core cost?",
    "Can I deduct a home office, and how much?",
    "Does QuickBooks integrate with Salesforce?",  # not in the KB
]
for q in DEMO_QUESTIONS:
    ans, _ = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}")

## Stretch goals
The base assistant above is what everyone ships. Pick up these in order as
time allows. Hints only here; the full code is in the solution notebook.

1. **Agentic RAG.** Give a smolagents `CodeAgent` two tools, `search_help`
   (wrap `retrieve`) and `invoice_total(subtotal, state_code)` that adds sales
   tax (CA 7.25, NY 4.0, TX 6.25, WA 6.5, FL 6.0 percent). Ask for a California
   invoice total on a 1200 dollar subtotal and let the agent search then
   compute.
2. **LLM-as-a-judge.** Load `data/golden_qa.json` and, for each pair, ask a
   judge model to return JSON `{"faithful": ..., "correct": ...}` comparing the
   RAG answer and context to the expected answer. Report the score.
3. **Compare chunking.** Rebuild the index with `chunk_size=200` and compare
   which source article each question retrieves against the default chunks.
4. **LoRA ticket router.** Load `data/support_tickets.json` and adapt Lab 3.2:
   put a LoRA adapter on `distilbert-base-uncased`
   (`AutoModelForSequenceClassification`, `target_modules=["q_lin", "v_lin"]`,
   `task_type="SEQ_CLS"`), train on the 16 example tickets, and route the four
   test tickets into billing, payroll, tax_filing, or integrations.

In [ ]:
# TODO (stretch): your experiments here.